# Investigate Embedded Boxes as Boundaries Issue

## Setup

### Imports

In [1]:
import logging
from math import isnan

import pytest

# Local functions and classes
from types_and_classes import *
from utilities import *
from contour_plotting import *
from debug_tools import *
from structure_set import *
from relations import *
from region_slice import empty_structure
from contour_plotting import plot_roi_slice


INFO:metrics.base:Registered calculator: minimum_margins (ContainmentMarginsCalculator)
INFO:metrics.base:Registered calculator: maximum_margin (MaximumMarginCalculator)
INFO:metrics.base:Registered calculator: minimum_distance (MinimumDistanceCalculator)


### Global Settings

In [2]:
%matplotlib inline

### b) Function to identify relationships one each slice

In [3]:
def get_slice_relations(structure_a, structure_b):
    slices_a = set(structure_a.region_table['SliceIndex'])
    slices_b = set(structure_b.region_table['SliceIndex'])
    used_slices = slices_a | slices_b

    mask_a = structure_a.region_table.SliceIndex.isin(used_slices) & ~structure_a.region_table.Empty
    mask_b = structure_b.region_table.SliceIndex.isin(used_slices) & ~structure_b.region_table.Empty

    regions_a = structure_a.region_table.loc[mask_a, ['SliceIndex', 'RegionSlice']].set_index('SliceIndex')
    regions_b = structure_b.region_table.loc[mask_b, ['SliceIndex', 'RegionSlice']].set_index('SliceIndex')
    regions = regions_a.join(regions_b, how='outer', lsuffix='_a', rsuffix='_b').sort_index()

    cumulative = DE27IM()
    slice_rows = []

    for slice_index, row in regions.iterrows():
        region_a = row['RegionSlice_a']
        region_b = row['RegionSlice_b']

        relation = DE27IM(region_a, region_b)
        relation_type = relation.identify_relation()

        cumulative.merge(relation)
        cumulative_type = cumulative.identify_relation()

        slice_rows.append({
            'slice_index': float(slice_index),
            'a_present': not empty_structure(region_a),
            'b_present': not empty_structure(region_b),
            'slice_relation_type': relation_type.relation_type if relation_type else None,
            'slice_relation_label': relation_type.label if relation_type else None,
            'slice_de27im_bits': relation.relation,
            'slice_de27im_int': relation.int,
            'cumulative_relation_type': cumulative_type.relation_type if cumulative_type else None,
            'cumulative_relation_label': cumulative_type.label if cumulative_type else None,
            'cumulative_de27im_bits': cumulative.relation,
            'cumulative_de27im_int': cumulative.int,
        })

    slice_relation_df = pd.DataFrame(slice_rows).sort_values('slice_index').reset_index(drop=True)
    slice_relation_df.set_index('slice_index', inplace=True)
    return slice_relation_df, cumulative

# Error Getting Borders instead of Contains

In [4]:
def shifted_embedded_boxes_example():
    slice_spacing = 0.2
    # Body structure defines slices in use
    body = make_vertical_cylinder(roi_num=0, radius=20, length=10, offset_z=0,
                                spacing=slice_spacing)
    # laterally shifted boxes
    outer_cube = make_box(roi_num=1, width=6, offset_x=0, offset_z=0,
                         spacing=slice_spacing)
    inner_cube = make_box(roi_num=2, width=3, offset_x=0, offset_z=0,
                         spacing=slice_spacing)
    # combine the contours
    slice_data = outer_cube + inner_cube + body
    return slice_data


slice_data = shifted_embedded_boxes_example()
embedded_box_structures = StructureSet(slice_data)
structure_a = embedded_box_structures.structures[1]
structure_b = embedded_box_structures.structures[2]
embedded_box_relation = structure_a.relate(structure_b)
embedded_box_relation_type = embedded_box_relation.identify_relation()

print(f'\nOuter Cube {embedded_box_relation_type.label} Inner Cube')



INFO:structure_set:Adding structure Structure_0 (0)


INFO:structure_set:Adding structure Structure_1 (1)
INFO:structure_set:Adding structure Structure_2 (2)
INFO:structure_set:Calculating relationships for 3 structures
INFO:structure_set:Calculated relationships between Structure_0 (ROI 0) and Structure_1 (ROI 1) as: is Partitioned by
INFO:structure_set:Calculated relationships between Structure_0 (ROI 0) and Structure_2 (ROI 2) as: Borders
INFO:structure_set:Calculated relationships between Structure_1 (ROI 1) and Structure_2 (ROI 2) as: Borders
INFO:structure_set:Calculating logical flags for relationships
INFO:structure_set:Logical flag calculation complete



Outer Cube Borders Inner Cube


In [5]:
get_slice_relations(structure_a, structure_b)

(             a_present  b_present slice_relation_type slice_relation_label  \
 slice_index                                                                  
 -3.1              True      False                None                 None   
 -3.0              True      False                None                 None   
 -2.8              True      False                None                 None   
 -2.6              True      False                None                 None   
 -2.4              True      False                None                 None   
 ...                ...        ...                 ...                  ...   
  2.4              True      False                None                 None   
  2.6              True      False                None                 None   
  2.8              True      False                None                 None   
  3.0              True      False                None                 None   
  3.1              True      False                No

# ERROR without Body structure the Z margin becomes incorrect

In [6]:
def shifted_embedded_boxes_example():
    slice_spacing = 0.2
    # Body structure defines slices in use
    body = make_vertical_cylinder(roi_num=0, radius=20, length=10, offset_z=0,
                                spacing=slice_spacing)
    # laterally shifted boxes
    outer_cube = make_box(roi_num=1, width=8, offset_x=0, offset_z=0,
                         spacing=slice_spacing)
    inner_cube = make_box(roi_num=2, width=2, offset_x=2, offset_y=-1, offset_z=1,
                         spacing=slice_spacing)
    # combine the contours
    slice_data = outer_cube + inner_cube + body
    return slice_data


slice_data = shifted_embedded_boxes_example()
embedded_box_structures = StructureSet(slice_data)
structure_a = embedded_box_structures.structures[1]
structure_b = embedded_box_structures.structures[2]
relation = structure_a.relate(structure_b)
relation_type = relation.identify_relation()

print(f'\nOuter Cube {relation_type.label} Inner Cube')

# Calculate margins between the two boxes (ROI 1 and ROI 2)
margin_result = embedded_box_structures.calculate_metric(1, 2, 'minimum_margins')

# Display results
print(f"Orthogonal Margins: {margin_result.orthogonal_margins}")
print(f"Minimum Margin: {margin_result.minimum_margin}")


INFO:structure_set:Adding structure Structure_0 (0)
INFO:structure_set:Adding structure Structure_1 (1)
INFO:structure_set:Adding structure Structure_2 (2)


INFO:structure_set:Calculating relationships for 3 structures
INFO:structure_set:Calculated relationships between Structure_0 (ROI 0) and Structure_1 (ROI 1) as: Contains
INFO:structure_set:Calculated relationships between Structure_0 (ROI 0) and Structure_2 (ROI 2) as: Contains
INFO:structure_set:Calculated relationships between Structure_1 (ROI 1) and Structure_2 (ROI 2) as: Contains
INFO:structure_set:Calculating logical flags for relationships
INFO:structure_set:Logical flag calculation complete
INFO:metrics.config:Loading metrics configuration from /workspaces/StructureRelations/src/metrics/metrics_config.json
INFO:metrics.config:Metrics configuration loaded successfully



Outer Cube Contains Inner Cube
Orthogonal Margins: {'x_neg': 5.0, 'x_pos': 1.0, 'y_neg': 2.0, 'y_pos': 4.0, 'z_neg': 4.0, 'z_pos': 2.0}
Minimum Margin: 1.0


In [9]:
def shifted_embedded_boxes_no_body_example():
    slice_spacing = 0.2
    # Body structure defines slices in use
    body = make_vertical_cylinder(roi_num=0, radius=20, length=10, offset_z=0,
                                spacing=slice_spacing)
    # laterally shifted boxes
    outer_cube = make_box(roi_num=1, width=8, offset_x=0, offset_z=0,
                         spacing=slice_spacing)
    inner_cube = make_box(roi_num=2, width=2, offset_x=2, offset_y=-1, offset_z=1,
                         spacing=slice_spacing)
    # combine the contours
    slice_data = outer_cube + inner_cube #+ body
    return slice_data


slice_data = shifted_embedded_boxes_no_body_example()
embedded_box_structures = StructureSet(slice_data)
structure_a = embedded_box_structures.structures[1]
structure_b = embedded_box_structures.structures[2]
relation = structure_a.relate(structure_b)
relation_type = relation.identify_relation()

print(f'\nOuter Cube {relation_type.label} Inner Cube')

# Calculate margins between the two boxes (ROI 1 and ROI 2)
margin_result = embedded_box_structures.calculate_metric(1, 2, 'minimum_margins')

# Display results
print(f"Orthogonal Margins: {margin_result.orthogonal_margins}")
print(f"Minimum Margin: {margin_result.minimum_margin}")


INFO:structure_set:Adding structure Structure_1 (1)


INFO:structure_set:Adding structure Structure_2 (2)
INFO:structure_set:Calculating relationships for 2 structures
INFO:structure_set:Calculated relationships between Structure_1 (ROI 1) and Structure_2 (ROI 2) as: Contains
INFO:structure_set:Calculating logical flags for relationships
INFO:structure_set:Logical flag calculation complete



Outer Cube Contains Inner Cube
Orthogonal Margins: {'x_neg': 5.0, 'x_pos': 1.0, 'y_neg': 2.0, 'y_pos': 4.0, 'z_neg': 3.95, 'z_pos': 1.95}
Minimum Margin: 1.0


## Debugging code

In [ ]:
# Create figure and plot
fig, ax = plt.subplots(2, 2, figsize=(7, 2))
slice_idx = 3.0
plot_roi_slice(structures, slice_index=slice_idx,
               roi_list=[1, 2],
               add_axis=True, axes=ax[0,0], tolerance=tolerance,
               plot_mode='relationship', show_legend=False)
slice_idx = 3.5
plot_roi_slice(structures, slice_index=slice_idx,
               roi_list=[1, 2],
               add_axis=True, axes=ax[0,1], tolerance=tolerance,
               plot_mode='relationship', show_legend=False)
slice_idx = 4.0
plot_roi_slice(structures, slice_index=slice_idx,
               roi_list=[1, 2],
               add_axis=True, axes=ax[1,0], tolerance=tolerance,
               plot_mode='relationship', show_legend=False)
slice_idx = 5.0
plot_roi_slice(structures, slice_index=slice_idx,
               roi_list=[1, 2],
               add_axis=True, axes=ax[1,1], tolerance=tolerance,
               plot_mode='relationship', show_legend=False)
plt.tight_layout()
plt.show()


In [ ]:
slices = structures.slice_sequence.sequence
slices[~slices.Original]
slices

Region pair (1, 0):
  Slice -5.0 to -3.5: 1.50 cm
  Slice  5.0 to  3.5: 1.50 cm

Region pair (2, 0):
  Slice -5.5 to -3.5: 2.00 cm
  Slice -5.0 to -3.5: 1.50 cm
  Slice 5.0 to 3.5: 1.50 cm
  Slice 5.5 to 3.5: 2.00 cm

In [ ]:
# Examine per-slice distances for detailed analysis
if distance_result.slice_distances:
    print("Per-Slice Distances:")
    for region_pair, slice_dists in distance_result.slice_distances.items():
        print(f"\nRegion pair {region_pair}:")
        for (slice_a, slice_b), dist in sorted(slice_dists.items()):
            print(f"  Slice {slice_a} to {slice_b}: {dist:.2f} cm")

In [ ]:
# Display results
print(f"Minimum Distance: {distance_result.minimum_distance:.2f} cm")
print(f"Closest Region Pair: {distance_result.closest_region_pair}")
print(f"Closest Slice: {distance_result.closest_slice}")

# Per-region distances (if multi-region structures)
if distance_result.per_region_minimum_distance:
    print("\nPer-Region Distances:")
    for region_pair, dist in distance_result.per_region_minimum_distance.items():
        print(f"  Region pair {region_pair}: {dist:.2f} cm")


In [ ]:
structure_a = structures.structures[1]
structure_b = structures.structures[2]
slice_relation_df, cumulative = get_slice_relations(structure_a, structure_b)